# Session 2, Block 1 &mdash; Pandas, Tidy Data & Entity Matching
**Data Science Techniques and Real-World Applications &mdash; WS 2026**

Thursday 10 September 2026 &middot; Block 1 (09:30&ndash;11:00)

Today's dataset: daily prices and returns for four tech stocks (2018&ndash;2024), plus a small
company-info table we'll build ourselves. This block sets up the skills **CS1** needs directly.


## <span style="color:#1e3a8a">Tidy data</span>

Before touching any code: what makes a dataset easy to work with?

A **tidy** dataset is organized so that:
1. Each observation forms a **row**.
2. Each variable forms a **column**.
3. Each kind of observation forms its own **table**.
4. In relational datasets, the link between tables is clear and documented.

**Types of variables**
- **Quantitative** &mdash; born as numbers (integers, floats). *Flow* variables are measured over a
  time frame (e.g. monthly sales); *stock* variables are measured at a point in time (e.g. a closing price).
- **Qualitative / categorical** &mdash; labels, or numbers used as labels (e.g. `0 = US, 1 = EU, 2 = RoW`).
  A **binary** variable (0/1) is a special case.

**Types of observations**

| Observation type | Meaning | Example |
|---|---|---|
| Cross-sectional (xsec) | different units observed at the same time | 4 companies' closing prices on one day |
| Time series (tseries) | one unit observed over time | AAPL's daily price, 2018&ndash;2024 |
| Panel (xt / longitudinal) | many units observed over time | our full tech-stocks dataset: 4 tickers &times; many days |


## <span style="color:#1e3a8a">Pandas essentials</span>

Pandas is the standard library for tabular data in Python &mdash; a DataFrame is essentially a tidy
data table with row and column labels.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("data/techstocks.xlsx", sheet_name="main")
df.head()

What are we looking at? One row per (ticker, day) &mdash; a panel, per the table above.

In [ ]:
df.shape

In [ ]:
df.dtypes

### Indexing & filtering

In [ ]:
# a single column -> a Series; unique values in it
df["Ticker Symbol"].unique()

In [ ]:
# filter to one ticker
aapl = df[df["Ticker Symbol"] == "AAPL"].reset_index(drop=True)
aapl.head()

In [ ]:
# multiple conditions: AAPL, after 2020, with an above-average price
aapl_recent = df[
    (df["Ticker Symbol"] == "AAPL")
    & (df["Names Date"] > pd.to_datetime("2020-01-01"))
    & (df["Price or Bid/Ask Average"] > df["Price or Bid/Ask Average"].mean())
]
aapl_recent.head()

### Basic analytics

In [ ]:
aapl["Returns"].describe()

A self-defined function (or a `lambda`) applied with `.map()` works the same way as it does
on a plain Python list, just column-by-column:

In [ ]:
aapl["%returns"] = aapl["Returns"].map(lambda x: x * 100)
aapl[["Names Date", "Returns", "%returns"]].head()

### GroupBy

In [ ]:
groups = df.groupby("Ticker Symbol")
groups["Returns"].describe()

In [ ]:
# any function works, including your own
groups["Returns"].aggregate([np.mean, np.std, "count"])

<span style="color:#b45309">**Exercise 1: Which ticker has the most positive-return days?**</span>

Using `groups` from above, find which ticker has the largest number of days with a positive return (`Returns > 0`).

Hint: `(x > 0).sum()` inside `.aggregate(lambda x: ...)` counts positive values in each group; `.idxmax()` on the resulting Series gives you the group with the largest count.

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">Merging</span>

**Merge types**, depending on how keys match between two tables:

- **1:1** &mdash; each key appears once in each table (e.g. one row per country in two tables).
- **1:m** (one-to-many) &mdash; a key appears once in the "one" table, multiple times in the "many"
  table (e.g. one company row, many days of prices).
- **m:m** (many-to-many) &mdash; rare, usually a sign your keys aren't well defined yet. Be careful:
  it silently produces a much bigger table than you expect (every match on the left pairs with every
  match on the right).

And **how** two tables combine when a key doesn't appear on both sides:

| `how=` | Keeps |
|---|---|
| `"inner"` | only keys present in *both* tables |
| `"left"` | all keys from the left table, `NaN` where the right has no match |
| `"right"` | all keys from the right table, `NaN` where the left has no match |
| `"outer"` | every key from either table, `NaN` wherever a side is missing |

Let's build a second, small table to merge with `df` &mdash; company info, indexed by ticker
(a classic **1:many** merge: one company row matches many price rows):


In [ ]:
company_info = pd.DataFrame({
    "Ticker Symbol": ["AAPL", "GOOG", "AMZN", "NFLX"],
    "Sector":        ["Technology", "Technology", "Consumer Discretionary", "Communication Services"],
    "Headquarters":  ["Cupertino, US", "Mountain View, US", "Seattle, US", "Los Gatos, US"],
    "Founded":       [1976, 1998, 1994, 1997],
})
company_info

In [ ]:
merged = df.merge(company_info, on="Ticker Symbol", how="left")
merged.head()

Always validate what kind of merge you *think* you're doing &mdash; `validate` raises an error
if your assumption is wrong, before a silently-wrong merge corrupts your analysis:

In [ ]:
merged_checked = df.merge(company_info, on="Ticker Symbol", how="left", validate="m:1")
print("Merge validated: many rows in df matched at most one row in company_info.")

<span style="color:#b45309">**Exercise 2: A one-to-one merge**</span>

Build a small DataFrame `latest_price` with one row per ticker (its most recent closing price -- `df.sort_values("Names Date").groupby("Ticker Symbol").tail(1)` gets you there). Merge it with `company_info` &mdash; this time it should be a genuine **1:1** merge. Validate it as such.

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">Regex essentials</span>

A **regular expression** is a search pattern for text. Python's built-in `re` module is the tool;
useful when a company name, ticker, or date shows up embedded in messier text than a clean column.


In [ ]:
import re

text = "Q3 2024 revenue grew 12%, driven by AAPL and GOOG. Contact: ir@example.com"

Three core functions:
- `re.search(pattern, text)` &mdash; first match anywhere in the text
- `re.findall(pattern, text)` &mdash; *all* matches, as a list
- `re.sub(pattern, replacement, text)` &mdash; find and replace


In [ ]:
re.search(r"AAPL", text)

In [ ]:
# \d = a digit, + = one or more -> matches runs of digits
re.findall(r"\d+", text)

In [ ]:
# \w = alphanumeric character, common pattern for a simple email
re.findall(r"[\w.-]+@[\w.-]+", text)

In [ ]:
# character classes: match either of two tickers
re.findall(r"AAPL|GOOG", text)

In [ ]:
# anonymize: replace every all-caps ticker-looking token
re.sub(r"\b[A-Z]{2,5}\b", "[TICKER]", text)

<span style="color:#b45309">**Exercise 3: Extract every 4-digit year from a headline**</span>

From `"London Olympics 2012 was followed by Rio 2016 and Tokyo 2021."`, extract every 4-digit year using `re.findall`.

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">Fuzzy matching &mdash; for CS1</span>

Merging works perfectly when both tables share a clean key. In the real world, they often don't:
the *same* company can appear as `"W W INTERNATIONAL INC"`, `"WW International, Inc."`, and
`"WW International"` across two datasets. **CS1** is built entirely around exactly this problem.

**Fuzzy matching** scores how similar two strings are, so you can match on "close enough" rather
than "identical." The `thefuzz` library implements this via the **Levenshtein distance** (roughly:
how many single-character edits turn one string into the other).


In [ ]:
from thefuzz import fuzz, process

In [ ]:
a = "WW International Inc"
b = "W W INTERNATIONAL, INC."

print("ratio:          ", fuzz.ratio(a, b))                # plain character-level similarity
print("token_sort_ratio:", fuzz.token_sort_ratio(a, b))     # ignores word order
print("token_set_ratio: ", fuzz.token_set_ratio(a, b))      # ignores word order AND duplicate/extra words

`token_sort_ratio` and `token_set_ratio` matter because company names get reordered or
padded ("Inc", "Ltd", "Corp") in ways plain `ratio` is too literal to see past. `WRatio` (weighted
ratio) combines several of these heuristics and is a reasonable default when you're not sure which
to pick.

In [ ]:
print(fuzz.WRatio(a, b))

**`process.extractOne`** finds the best match for one string out of a whole list of candidates
&mdash; exactly what you need to fuzzy-merge one dataset's names against another's:

In [ ]:
candidates = ["Apple Inc", "Alphabet Inc", "Amazon.com Inc", "Netflix Inc", "WW International Inc"]

process.extractOne("W.W. Internatinal, Inc", candidates, scorer=fuzz.token_sort_ratio)

<span style="color:#b45309">**Exercise 4: Fuzzy-merge two small tables**</span>

Two DataFrames below share the same real-world entities under slightly different spellings. For each `Key` in `df1`, find its best-matching `Key` in `df2` (use `process.extractOne`), store it in a new column `MergeKey`, then merge `df1` and `df2` on that key. 

How many of the 4 rows in `df1` found a sensible match in `df2`?

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


### Looking ahead to CS1

CS1 asks you to link earnings-call records to stock-return records using exactly this toolkit &mdash;
except the "obvious" match isn't always the *right* one (a high fuzzy score doesn't mean the merge is
correct: company name changes, M&amp;A, and near-duplicate names can all fool a similarity score).
Treat a fuzzy match as a **candidate** to validate, not a guaranteed answer &mdash; the case brief asks
you directly what could go wrong and how you'd check.
